# NB4 — Gwet's AC Analysis

In [ ]:
import os

# Mount Google Drive
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

import pandas as pd
import numpy as np

# Make sure output folders exist
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/graphs", exist_ok=True)

MASTER_PATH = "Master_Eval_Sheet.xlsx"

# ── Column names ────────────────────────────────────────────────
# Master sheet structure:
#   Row 1 = section labels  (Gold Reference + Annotation, H1-Ashish ...)
#   Row 2 = column names    (#, Sentence ID, ..., C1,C2,C3,C4,Total repeating)
#   Row 3 onwards = data
# We skip both header rows and assign unambiguous column names directly.

col_names = [
    "#", "Sentence_ID", "Source_File", "Source_Sentence", "Gold_Category",
    "LLM", "Has_Errors", "Error_Span", "Annotated_Category", "Description",
    "Corrected_Sentence",
    "H1_C1", "H1_C2", "H1_C3", "H1_C4", "H1_Total",
    "H2_C1", "H2_C2", "H2_C3", "H2_C4", "H2_Total",
    "L1_C1", "L1_C2", "L1_C3", "L1_C4", "L1_Total",
    "L2_C1", "L2_C2", "L2_C3", "L2_C4", "L2_Total",
    "L3_C1", "L3_C2", "L3_C3", "L3_C4", "L3_Total",
    "L4_C1", "L4_C2", "L4_C3", "L4_C4", "L4_Total",
]

df = pd.read_excel(
    MASTER_PATH,
    sheet_name="Master Eval Sheet",
    header=None,
    skiprows=2,
    names=col_names
)

# Drop empty trailing rows and the summary TOTAL row
df = df[df["#"].notna()].copy()
df = df[df["#"].astype(str) != "TOTAL"].copy()
df = df.reset_index(drop=True)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample:")
print(df[["Sentence_ID", "Gold_Category", "LLM", "H1_C1", "H2_C1", "L1_C1", "L2_C1", "L3_C1", "L4_C1"]].head(3))

# ── Rater groups ────────────────────────────────────────────────
G1_HUMANS           = ["H1", "H2"]
G2_ANNOTATOR_LLMS   = ["L1", "L2"]
G3_NON_ANNOTATOR    = ["L3", "L4"]
G4_ALL_LLMS         = ["L1", "L2", "L3", "L4"]
G5_HUMANS_ANN       = ["H1", "H2", "L1", "L2"]
G6_HUMANS_NONANN    = ["H1", "H2", "L3", "L4"]
G7_ALL              = ["H1", "H2", "L1", "L2", "L3", "L4"]

ALL_GROUPS = {
    "G1_Humans":             G1_HUMANS,
    "G2_Annotator_LLMs":     G2_ANNOTATOR_LLMS,
    "G3_NonAnnotator_LLMs":  G3_NON_ANNOTATOR,
    "G4_All_LLMs":           G4_ALL_LLMS,
    "G5_Humans+Ann_LLMs":    G5_HUMANS_ANN,
    "G6_Humans+NonAnn_LLMs": G6_HUMANS_NONANN,
    "G7_All":                G7_ALL,
}

CATEGORIES = ["Script Normalization", "Spelling & Typographical Error",
              "Grammatical Error", "Code-Mixing / Wrong Language",
              "Correct Sentence / No Errors"]

TASKS = ["C1", "C2", "C3", "C4"]

print("\nRater groups and tasks defined.")


In [ ]:
# ── Cell 2: Gwet AC1/AC2 — manual implementation (no irrCAC) ──
# irrCAC has binary incompatibility with Colab's numpy 2.x.
# We implement Gwet's AC1 and AC2 directly from the formula.
# Reference: Gwet (2014) Handbook of Inter-Rater Reliability.

import numpy as np
import pandas as pd
from itertools import combinations

def compute_gwet_ac(data, raters, task, weighted=False):
    cols = [r + "_" + task for r in raters]
    ratings = data[cols].values

    # Derive categories from actual data values
    flat_all = ratings.flatten().astype(float)
    flat_all = flat_all[~np.isnan(flat_all)]
    categories = sorted(list(set(int(v) for v in flat_all)))
    q = len(categories)

    if q < 2:
        return None

    # Build weight matrix
    W = np.zeros((q, q))
    for i in range(q):
        for j in range(q):
            if weighted:
                W[i][j] = 1 - abs(categories[i] - categories[j]) / (q - 1)
            else:
                W[i][j] = 1.0 if i == j else 0.0

    # Per-item observed agreement
    pa_list = []
    for row in ratings:
        valid = [int(v) for v in row.astype(float) if not np.isnan(v)]
        if len(valid) < 2:
            pa_list.append(np.nan)
            continue
        pairs = list(combinations(valid, 2))
        item_agree = np.mean([
            W[categories.index(a)][categories.index(b)]
            for a, b in pairs
        ])
        pa_list.append(item_agree)

    pa = np.nanmean(pa_list)

    # Category proportions
    flat = [int(v) for v in ratings.flatten().astype(float) if not np.isnan(v)]
    pk = np.array([flat.count(c) / len(flat) for c in categories])

    # Gwet's chance agreement
    pe = sum(W[i][j] * pk[i] * pk[j] for i in range(q) for j in range(q))

    if pe >= 1.0:
        return None

    ac = (pa - pe) / (1 - pe)
    return round(ac, 4)

# Sanity check on small example
test_df = pd.DataFrame({
    "H1_C1": [1, 0, 1, 1, 0, 1],
    "H2_C1": [1, 0, 1, 0, 0, 1],
})
check = compute_gwet_ac(test_df, ["H1", "H2"], "C1", weighted=False)
print("Sanity check (expect ~0.66):", check)
print("Helper function ready.")


In [ ]:
# ── Cell 3: Function defined in Cell 2 — nothing to do here ───
# compute_gwet_ac is already defined above with the manual implementation.
# This cell is kept as a placeholder to maintain cell numbering.
print("compute_gwet_ac is ready from Cell 2.")


In [ ]:
# ── Cell 4: Overall Gwet AC (all 100 rows) ────────────────────

overall_results = []

for group_name, raters in ALL_GROUPS.items():
    for task in TASKS:
        # AC1 — unweighted, nominal, for all tasks
        ac_unweighted = compute_gwet_ac(df, raters, task, weighted=False)

        # AC2 — weighted, ordinal, only for C2 and C4
        if task in ["C2", "C4"]:
            ac_weighted = compute_gwet_ac(df, raters, task, weighted=True)
        else:
            ac_weighted = None

        overall_results.append({
            "Group": group_name,
            "Task": task,
            "N_Raters": len(raters),
            "N_Items": len(df),
            "Gwet_AC1_Unweighted": ac_unweighted,
            "Gwet_AC2_Weighted": ac_weighted,
        })

overall_df = pd.DataFrame(overall_results)
print("Overall Gwet AC:")
print(overall_df.to_string(index=False))


In [ ]:
# ── Cell 5: Category-wise Gwet AC ────────────────────────────

category_results = []

for category in CATEGORIES:
    cat_df = df[df["Gold_Category"] == category].copy()

    for group_name, raters in ALL_GROUPS.items():
        for task in TASKS:
            ac_unweighted = compute_gwet_ac(cat_df, raters, task, weighted=False)

            if task in ["C2", "C4"]:
                ac_weighted = compute_gwet_ac(cat_df, raters, task, weighted=True)
            else:
                ac_weighted = None

            category_results.append({
                "Category": category,
                "Group": group_name,
                "Task": task,
                "N_Raters": len(raters),
                "N_Items": len(cat_df),
                "Gwet_AC1_Unweighted": ac_unweighted,
                "Gwet_AC2_Weighted": ac_weighted,
            })

category_df = pd.DataFrame(category_results)
print("Category-wise Gwet AC (first 20 rows):")
print(category_df.head(20).to_string(index=False))


In [ ]:
# ── Cell 6: Save results ──────────────────────────────────────

output_path = "outputs/tables/NB4_Gwet_AC_Results.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    overall_df.to_excel(writer, sheet_name="Overall", index=False)
    category_df.to_excel(writer, sheet_name="By_Category", index=False)

print("Saved:", output_path)
